# Real-Time Web Search with Mistral Using free-web-search-ultimate

This notebook demonstrates how to give Mistral models real-time web search capabilities using the [free-web-search-ultimate](https://github.com/wd041216-bit/free-web-search-ultimate) Python package — **completely free, no additional API keys required**.

[![free-web-search-ultimate](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate/badges/score.svg)](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)

## What you'll learn

- How to use `free-web-search-ultimate` as a function calling tool for Mistral
- How to build a research assistant with real-time web access
- How to handle tool use with Mistral's function calling API
- How to combine web search with Mistral's reasoning capabilities

## Why free-web-search-ultimate?

| Feature | Details |
|---------|--------|
| **Cost** | Completely free — no API key, no subscription |
| **Privacy** | Uses DuckDuckGo and privacy-respecting search engines |
| **Reliability** | Multiple search backends with automatic fallback |
| **MCP Support** | Works as an MCP server for Claude Desktop and other clients |
| **CLI Support** | Also works as a standalone CLI tool (`free-web-search`) |

In [ ]:
# Install required packages
%pip install free-web-search-ultimate mistralai --quiet

In [ ]:
import json
import os
from mistralai import Mistral
from free_web_search.search_web import UltimateSearcher

# Initialize the Mistral client
client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY", ""))

# Initialize the searcher
searcher = UltimateSearcher()


def web_search(
    query: str,
    max_results: int = 5,
    timelimit: str | None = None,
) -> str:
    """Search the web for current information.

    Args:
        query: The search query string.
        max_results: Maximum number of results to return. Defaults to 5.
        timelimit: Optional time filter ('d' for day, 'w' for week,
            'm' for month, 'y' for year).

    Returns:
        A formatted string containing search results with titles, URLs,
        and snippets.
    """
    results = searcher.search(query, max_results=max_results, timelimit=timelimit)
    if not results:
        return "No results found."
    output = []
    for r in results:
        output.append(f"Title: {r.get('title', 'N/A')}")
        output.append(f"URL: {r.get('url', 'N/A')}")
        output.append(f"Snippet: {r.get('body', 'N/A')}")
        output.append("")
    return "\n".join(output)


def news_search(
    query: str,
    max_results: int = 5,
    timelimit: str | None = "w",
) -> str:
    """Search for recent news articles.

    Args:
        query: The search query string.
        max_results: Maximum number of results to return. Defaults to 5.
        timelimit: Time filter for news recency ('d' for day, 'w' for week,
            'm' for month). Defaults to 'w' (past week).

    Returns:
        A formatted string containing news results with titles, URLs,
        and snippets.
    """
    results = searcher.search_news(query, max_results=max_results, timelimit=timelimit)
    if not results:
        return "No news found."
    output = []
    for r in results:
        output.append(f"Title: {r.get('title', 'N/A')}")
        output.append(f"URL: {r.get('url', 'N/A')}")
        output.append(f"Snippet: {r.get('body', 'N/A')}")
        output.append("")
    return "\n".join(output)


## Build the Research Assistant

In [ ]:
def research_with_mistral(
    question: str,
    model: str = "mistral-large-latest",
    max_iterations: int = 10,
) -> str:
    """Run a research query using Mistral with web search tools.

    Args:
        question: The research question to answer.
        model: The Mistral model to use. Defaults to 'mistral-large-latest'.
        max_iterations: Maximum number of tool-call iterations to prevent
            infinite loops. Defaults to 10.

    Returns:
        The final answer string from Mistral.
    """
    tools = [
        {
            "type": "function",
            "function": {
                "name": "web_search",
                "description": (
                    "Search the web for current information. Use for general queries "
                    "and topics that may have changed recently."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "The search query"},
                        "max_results": {
                            "type": "integer",
                            "description": "Maximum number of results (default: 5)",
                        },
                        "timelimit": {
                            "type": "string",
                            "description": "Time filter: 'd' (day), 'w' (week), 'm' (month), 'y' (year)",
                        },
                    },
                    "required": ["query"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "news_search",
                "description": (
                    "Search for recent news articles. Use for current events "
                    "and time-sensitive topics."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "The news search query"},
                        "max_results": {
                            "type": "integer",
                            "description": "Maximum number of results (default: 5)",
                        },
                        "timelimit": {
                            "type": "string",
                            "description": "Time filter: 'd' (day), 'w' (week), 'm' (month)",
                        },
                    },
                    "required": ["query"],
                },
            },
        },
    ]

    messages = [{"role": "user", "content": question}]

    for iteration in range(max_iterations):
        response = client.chat.complete(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(msg)
            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                if fn_name == "web_search":
                    result = web_search(**fn_args)
                elif fn_name == "news_search":
                    result = news_search(**fn_args)
                else:
                    result = f"Unknown tool: {fn_name}"
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "name": fn_name,
                    "content": result,
                })
        else:
            return msg.content

    return f"Reached maximum iterations ({max_iterations}) without a final answer."


## Example 1: Current Events Research

In [ ]:
# Set to True to run the live search examples below.
# Leave as False to avoid outbound network requests on fresh checkout.
RUN_EXAMPLES = False


In [ ]:
if RUN_EXAMPLES:
    # Example: Ask about current AI developments
    result = research_with_mistral(
        "What are the latest developments in AI language models in 2025?"
    )

## Example 2: News Research

In [ ]:
if RUN_EXAMPLES:
    # Example: Research recent news
    result = research_with_mistral(
        "What are the latest news about open source AI models?"
    )

## Example 3: Technical Research

In [ ]:
if RUN_EXAMPLES:
    # Example: Technical research
    result = research_with_mistral(
        "What is the Model Context Protocol (MCP) and which AI tools support it?"
    )

## Using as MCP Server

`free-web-search-ultimate` also works as an MCP (Model Context Protocol) server, allowing direct integration with Claude Desktop, Cursor, and other MCP-compatible clients:

```json
{
  "mcpServers": {
    "free-web-search": {
      "command": "free-web-search-mcp"
    }
  }
}
```

## CLI Usage

```bash
# Web search
search-web "latest Mistral AI news"

# News search
search-web --type news "AI breakthroughs 2025"
```

## Resources

- [GitHub Repository](https://github.com/wd041216-bit/free-web-search-ultimate)
- [PyPI Package](https://pypi.org/project/free-web-search-ultimate/)
- [Glama MCP Server](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)
- [Mistral Function Calling Docs](https://docs.mistral.ai/capabilities/function_calling/)